# Data prep — Baltic shadow-fleet detection

Fetches the two layers the detection pipeline needs and checks they line up:

- **Sentinel-1 SAR** (ship imagery) — via openEO / Copernicus Data Space.
- **DMA AIS** (ship broadcasts) — Danish daily archive. DMA's own host
  (web.ais.dk) is down, so we pull the identical raw zips from a Hugging Face
  mirror that covers **September 2025**.

AOI = open water around **Bornholm** (dense tanker traffic, near subsea cable /
pipeline corridors, inside Danish AIS coverage). Window = mid-Sept 2025.

Run top to bottom. Pick an acquisition date in **Step 2**, set it in **Step 3**.

In [1]:
from pathlib import Path

# Bounding box (lon/lat, WGS84) — main shipping lane NW of Bornholm, placed on the
# AIS traffic density (median big-vessel sits at ~14.69 E, 55.34 N).
AOI = {"west": 14.4, "south": 55.2, "east": 15.2, "north": 55.6}

# Window to search for a Sentinel-1 pass. Sept 2025: the DMA AIS host web.ais.dk
# is currently down, so we use a Hugging Face mirror that holds Sept-2025 dailies.
# (Once web.ais.dk is back, any date with both an S1 pass and DMA AIS works.)
TEMPORAL_EXTENT = ["2025-09-12", "2025-09-19"]

# Minimum length (m) for an AIS-mandated vessel; size gate + AIS pre-filter.
MIN_VESSEL_LENGTH_M = 100

DATA_DIR = Path("data/bornholm")
S1_PATH = DATA_DIR / "s1_grd.nc"
AIS_RAW_DIR = DATA_DIR / "ais_raw"
AIS_FILTERED_PATH = DATA_DIR / "ais_filtered.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Fetch Sentinel-1 SAR

First run prints a URL + code: visit it and log in with your Copernicus Data
Space credentials. The token caches, so later runs authenticate silently.
`sar_backscatter` turns raw GRD into calibrated sigma0; the result is a
georeferenced datacube (lon/lat coords) just like the org's prep notebook.

In [2]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

cube = connection.load_collection(
    "SENTINEL1_GRD",
    spatial_extent=AOI,
    temporal_extent=TEMPORAL_EXTENT,
    bands=["VV", "VH"],
)
cube = cube.sar_backscatter(coefficient="sigma0-ellipsoid")

cube.execute_batch(title="bornholm-s1-grd", outputfile=str(S1_PATH))
print("saved", S1_PATH)

Authenticated using refresh token.
0:00:00 Job 'j-26060219540841e1b099e8e35c9ddb15': send 'start'
0:00:33 Job 'j-26060219540841e1b099e8e35c9ddb15': queued (progress 0%)
0:00:38 Job 'j-26060219540841e1b099e8e35c9ddb15': queued (progress 0%)
0:00:44 Job 'j-26060219540841e1b099e8e35c9ddb15': queued (progress 0%)
0:00:52 Job 'j-26060219540841e1b099e8e35c9ddb15': queued (progress 0%)
0:01:02 Job 'j-26060219540841e1b099e8e35c9ddb15': queued (progress 0%)
0:01:15 Job 'j-26060219540841e1b099e8e35c9ddb15': running (progress N/A)
0:01:30 Job 'j-26060219540841e1b099e8e35c9ddb15': running (progress N/A)
0:01:49 Job 'j-26060219540841e1b099e8e35c9ddb15': running (progress N/A)
0:02:14 Job 'j-26060219540841e1b099e8e35c9ddb15': running (progress N/A)
0:02:44 Job 'j-26060219540841e1b099e8e35c9ddb15': running (progress N/A)
0:03:21 Job 'j-26060219540841e1b099e8e35c9ddb15': running (progress N/A)
0:04:08 Job 'j-26060219540841e1b099e8e35c9ddb15': running (progress N/A)
0:05:06 Job 'j-26060219540841e1b099e

## Step 2 — Which dates did we get?

The cube spans the whole window (typically 2–3 passes). Pick one acquisition
date below and use it in Step 3 — the AIS must be for the **same day**.

In [4]:
import xarray as xr

ds = xr.open_dataset(S1_PATH)

print("t attrs:", ds["t"].attrs)
print("VV attrs:", ds["VV"].attrs)
print("global attrs keys:", list(ds.attrs)[:20])
print(ds.attrs.get("title"), ds.attrs.get("history"))

time_dim = "t" if "t" in ds.dims else "time"
print("dims:", dict(ds.sizes), "| bands:", list(ds.data_vars))
print("acquisitions:")
for value in ds[time_dim].values:
    print("  ", str(value)[:19])

t attrs: {'standard_name': 't', 'long_name': 't', 'axis': 'T'}
VV attrs: {'long_name': 'VV', 'units': '', 'grid_mapping': 'crs'}
global attrs keys: ['Conventions', 'institution', 'description', 'title']
 None
dims: {'t': 6, 'x': 3840, 'y': 3896} | bands: ['crs', 'VV', 'VH']
acquisitions:
   2025-09-12T00:00:00
   2025-09-13T00:00:00
   2025-09-14T00:00:00
   2025-09-15T00:00:00
   2025-09-17T00:00:00
   2025-09-18T00:00:00


## Step 2b — Recover the true overpass times

openEO labels each pass at midnight, but the matcher needs the real sensing
instant. We query the Copernicus product catalogue (no auth) for the S1 GRD
products over the AOI in our window and build a `date -> sensing_start` map.

In [5]:
import pandas as pd
import requests

# AOI centre point used to intersect the catalogue.
lon = (AOI["west"] + AOI["east"]) / 2
lat = (AOI["south"] + AOI["north"]) / 2

odata = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
flt = (
    "Collection/Name eq 'SENTINEL-1' "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;POINT({lon} {lat})') "
    f"and ContentDate/Start gt {TEMPORAL_EXTENT[0]}T00:00:00.000Z "
    f"and ContentDate/Start lt {TEMPORAL_EXTENT[1]}T00:00:00.000Z "
    "and contains(Name,'GRD')"
)
resp = requests.get(
    odata,
    params={"$filter": flt, "$orderby": "ContentDate/Start", "$top": 50},
    timeout=60,
)
resp.raise_for_status()

rows = [
    {"sensing_start": p["ContentDate"]["Start"], "name": p["Name"]}
    for p in resp.json()["value"]
]
overpass = pd.DataFrame(rows)
overpass["sensing_start"] = pd.to_datetime(overpass["sensing_start"])
overpass["date"] = overpass["sensing_start"].dt.strftime("%Y-%m-%d")
# date -> true sensing instant (first product per date)
ACQ_TIME = overpass.groupby("date")["sensing_start"].first().to_dict()
print(overpass[["date", "sensing_start", "name"]].to_string(index=False))

      date                    sensing_start                                                                         name
2025-09-12 2025-09-12 16:43:59.901000+00:00 S1C_IW_GRDH_1SDV_20250912T164359_20250912T164424_004094_008236_AF45_COG.SAFE
2025-09-12 2025-09-12 16:43:59.901000+00:00     S1C_IW_GRDH_1SDV_20250912T164359_20250912T164424_004094_008236_3310.SAFE
2025-09-13 2025-09-13 05:24:53.153238+00:00 S1A_IW_GRDH_1SDV_20250913T052453_20250913T052518_060965_07983E_EDA6_COG.SAFE
2025-09-13 2025-09-13 05:24:53.153238+00:00     S1A_IW_GRDH_1SDV_20250913T052453_20250913T052518_060965_07983E_6C4A.SAFE
2025-09-13 2025-09-13 16:36:45.980150+00:00 S1A_IW_GRDH_1SDV_20250913T163645_20250913T163710_060972_079881_E5CA_COG.SAFE
2025-09-13 2025-09-13 16:36:45.980150+00:00     S1A_IW_GRDH_1SDV_20250913T163645_20250913T163710_060972_079881_ED92.SAFE
2025-09-14 2025-09-14 05:15:24.545000+00:00 S1C_IW_GRDH_1SDV_20250914T051524_20250914T051549_004116_0082DF_67B9_COG.SAFE
2025-09-14 2025-09-14 05:15:24.5

## Step 3 — Fetch + filter DMA AIS for the chosen day

Streams that day's archive from the Hugging Face mirror (~0.6 GB), reads it in
chunks, and keeps only rows inside the AOI from vessels ≥100 m. Set `DATE` to a
date from Step 2 (must be in **Sept 2025** while we're on the mirror).



In [ ]:
import zipfile

import pandas as pd
import requests

DATE = "2025-09-14"  # <-- set to an acquisition date from Step 2 (Sept 2025)
USECOLS = [
    "# Timestamp", "MMSI", "Latitude", "Longitude", "SOG", "COG",
    "Heading", "IMO", "Name", "Ship type", "Width", "Length", "Draught",
]

# DMA's own host (web.ais.dk) is down; this Hugging Face dataset mirrors the
# identical raw DMA daily zips for Sept 2025. If the date isn't Sept 2025, point
# this at web.ais.dk/aisdata once it is back up.
HF_REPO = ("https://huggingface.co/datasets/mark000071/"
           "EnvShip-Bench_An_Environment-Enhanced_Benchmark_for_Short-Term_"
           "Vessel_Trajectory_Prediction/resolve/main/DMA/data_raw/dma/incoming")
url = f"{HF_REPO}/{DATE[:7]}/aisdk-{DATE}.zip"

AIS_RAW_DIR.mkdir(parents=True, exist_ok=True)
local_zip = AIS_RAW_DIR / f"aisdk-{DATE}.zip"
if not local_zip.exists():
    print("downloading", url)
    with requests.get(url, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with local_zip.open("wb") as fh:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                fh.write(chunk)

archive = zipfile.ZipFile(local_zip)
csv_name = next(n for n in archive.namelist() if n.endswith(".csv"))

kept = []
with archive.open(csv_name) as fh:
    for chunk in pd.read_csv(fh, usecols=USECOLS, chunksize=500_000):
        in_box = chunk["Latitude"].between(AOI["south"], AOI["north"]) & chunk[
            "Longitude"
        ].between(AOI["west"], AOI["east"])
        kept.append(chunk[in_box & (chunk["Length"] >= MIN_VESSEL_LENGTH_M)])

ais = pd.concat(kept, ignore_index=True)
ais["timestamp"] = pd.to_datetime(ais["# Timestamp"], format="%d/%m/%Y %H:%M:%S")
ais = ais.drop(columns="# Timestamp")
ais.to_csv(AIS_FILTERED_PATH, index=False)
print(f"kept {len(ais):,} rows, {ais['MMSI'].nunique()} vessels -> {AIS_FILTERED_PATH}")

## Step 4 — AIS sanity check

If `rows` is near zero, the AOI has little big-vessel traffic that day — widen
`AOI` or shift toward the Øresund / Great Belt and re-run Steps 1 and 3.

In [ ]:
ais = pd.read_csv(AIS_FILTERED_PATH, parse_dates=["timestamp"])
print(f"{len(ais):,} rows, {ais['MMSI'].nunique()} vessels, "
      f"{ais['timestamp'].min()} .. {ais['timestamp'].max()}")
ais[["timestamp", "Name", "Ship type", "Length", "Latitude", "Longitude"]].head()

## Step 5 — Do the layers line up?

Snap each vessel to its position at the exact overpass instant (interpolating its
AIS track), gated on **freshness**: trust a vessel only if it has a fix within
3 min of the pass. Stale fixes extrapolate to ghost positions.

- **green** = fresh AIS → its circle should sit on a bright radar return (proof
  AIS and SAR co-register)
- **red** = stale AIS → position untrusted (don't expect a return under it)
- a bright return with no green circle = a **dark-vessel candidate**

Three cells: build `snap`, full-frame overlay, then a per-vessel zoom grid.

In [ ]:
import numpy as np
from pyproj import CRS, Transformer

# Real sensing instant for the chosen day (DMA timestamps are UTC, tz-naive).
acq_ts = pd.Timestamp(ACQ_TIME[DATE])
if acq_ts.tzinfo is not None:
    acq_ts = acq_ts.tz_localize(None)
FRESH = pd.Timedelta(minutes=3)  # AIS freshness gate


def at_overpass(g):
    """Interpolate one vessel's track to the overpass instant; report time gap."""
    g = g.sort_values("timestamp").drop_duplicates("timestamp").set_index("timestamp")
    gap_s = np.abs((g.index - acq_ts).total_seconds()).min()
    if acq_ts <= g.index.min():
        pos = g.iloc[0]
    elif acq_ts >= g.index.max():
        pos = g.iloc[-1]
    else:
        pos = (g[["Latitude", "Longitude"]]
               .reindex(g.index.union([acq_ts])).interpolate("index").loc[acq_ts])
    return pd.Series({"Latitude": float(pos["Latitude"]),
                      "Longitude": float(pos["Longitude"]),
                      "Name": g["Name"].iloc[0], "Length": float(g["Length"].iloc[0]),
                      "gap_s": float(gap_s)})


cols_for_snap = ["timestamp", "Latitude", "Longitude", "Name", "Length"]
present = ais.groupby("MMSI").filter(
    lambda g: (g["timestamp"] - acq_ts).abs().min() <= pd.Timedelta(minutes=15))
snap = present.groupby("MMSI")[cols_for_snap].apply(at_overpass).reset_index()
snap["fresh"] = snap["gap_s"] <= FRESH.total_seconds()

cube_crs = CRS.from_wkt(ds["crs"].attrs["crs_wkt"])
to_utm = Transformer.from_crs("EPSG:4326", cube_crs, always_xy=True)
snap["x"], snap["y"] = to_utm.transform(snap["Longitude"].values, snap["Latitude"].values)
print(f"acquisition {acq_ts} UTC | {len(snap)} vessels present, "
      f"{int(snap['fresh'].sum())} fresh (<=3 min)")

In [ ]:
import matplotlib.pyplot as plt

vv = ds["VV"].sel({time_dim: DATE})
vv_db = 10 * np.log10(vv.where(vv > 0))

fig, ax = plt.subplots(figsize=(10, 9))
vv_db.plot.imshow(ax=ax, cmap="gray", vmin=-25, vmax=0, add_colorbar=False)
for sub, color, lab in [(snap[snap.fresh], "lime", "fresh AIS"),
                        (snap[~snap.fresh], "red", "stale AIS")]:
    ax.scatter(sub["x"], sub["y"], s=130, facecolors="none", edgecolors=color,
               linewidths=1.6, label=f"{lab} ({len(sub)})")
ax.set_aspect("equal")
ax.set_title(f"VV (dB) {acq_ts} UTC — fresh vs stale AIS")
ax.legend(loc="upper right")
plt.show()

In [ ]:
# Per-vessel zoom: each fresh ship should show a bright return inside its circle.
half = 1000  # metres around each vessel
fresh = snap[snap.fresh].reset_index(drop=True)
y_desc = ds.y.values[0] > ds.y.values[-1]

cols = 3
rows = int(np.ceil(max(len(fresh), 1) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for i, (_, r) in enumerate(fresh.iterrows()):
    ax = axes[i // cols][i % cols]
    ysl = slice(r.y + half, r.y - half) if y_desc else slice(r.y - half, r.y + half)
    sub = vv.sel(x=slice(r.x - half, r.x + half), y=ysl)
    (10 * np.log10(sub.where(sub > 0))).plot.imshow(
        ax=ax, cmap="gray", vmin=-25, vmax=5, add_colorbar=False)
    ax.plot(r.x, r.y, "o", mfc="none", mec="lime", ms=20, mew=2)
    ax.set_title(f"{r['Name']} ({r['Length']:.0f} m)", fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")
for j in range(len(fresh), rows * cols):
    axes[j // cols][j % cols].axis("off")
plt.tight_layout()
plt.show()